In [48]:
from __future__ import annotations


class KripkiState:
    def __init__(self, truths: list[str], children: list[KripkiState]):
        self.truths = truths
        self.children = children
        self.validate_children()

    def validate_children(self):
        for child in self.children:
            if not isinstance(child, KripkiState):
                raise ValueError("Children must be of type KripkiState")
            if len(set(self.truths) - set(child.truths)) != 0:
                raise ValueError("Children must have at least the same truths")

    def reduce_over_children(self, pred) -> list[bool]:
        result: list[bool] = [pred(self)]
        for child in self.children:
            result += child.reduce_over_children(pred)
        return result


class Formula:
    def __repr__(self):
        raise NotImplementedError()

    def evaluate(self, state: KripkiState):
        raise NotImplementedError()


class Prop(Formula):
    def __init__(self, name: str):
        self.name = name

    def __repr__(self):
        return self.name

    def evaluate(self, state: KripkiState):
        return self.name in state.truths


class Neg(Formula):
    def __init__(self, phi: Formula):
        self.phi = phi

    def __repr__(self):
        return f"¬{self.phi}"

    def evaluate(self, state: KripkiState):
        phi_over_children = state.reduce_over_children(lambda s: self.phi.evaluate(s))
        return not any(phi_over_children)


class Or(Formula):
    def __init__(self, phi: Formula, psi: Formula):
        self.phi = phi
        self.psi = psi

    def __repr__(self):
        return f"({self.phi} ∨ {self.psi})"

    def evaluate(self, state: KripkiState):
        return self.phi.evaluate(state) or self.psi.evaluate(state)


class And(Formula):
    def __init__(self, phi: Formula, psi: Formula):
        self.phi = phi
        self.psi = psi

    def __repr__(self):
        return f"({self.phi} ∧ {self.psi})"

    def evaluate(self, state: KripkiState):
        return self.phi.evaluate(state) and self.psi.evaluate(state)


class Implies(Formula):
    def __init__(self, phi: Formula, psi: Formula):
        self.phi = phi
        self.psi = psi

    def __repr__(self):
        return f"({self.phi} → {self.psi})"

    def evaluate(self, state: KripkiState):
        phi_over_children = state.reduce_over_children(lambda s: self.phi.evaluate(s))
        psi_over_children = state.reduce_over_children(lambda s: self.psi.evaluate(s))
        phi_implies_psi_over_children = [
            # phi -> psi
            phi_over_children[i] <= psi_over_children[i]
            for i in range(len(phi_over_children))
        ]
        return all(phi_implies_psi_over_children)


class Iff(Formula):
    def __init__(self, phi: Formula, psi: Formula):
        self.phi = phi
        self.psi = psi
        
    def __repr__(self):
        return f"({self.phi} ↔ {self.psi})"
    
    def evaluate(self, state: KripkiState):
        x = And(Implies(self.phi, self.psi), Implies(self.psi, self.phi))
        return x.evaluate(state)

In [4]:
# p = Prop("p")
# q = Prop("q")
# 
# np = Neg(p)
# nq = Neg(q)
# nnp = Neg(np)
# 
# nnp__i__p = Implies(nnp, p)
# nnp__i__p____q = Implies(nnp__i__p, q)
# nnp__i__p____q____q = Implies(nnp__i__p____q, q)

In [5]:
# c1 = KripkiState(["p", "q"], [])
# c2 = KripkiState([], [c1])

In [6]:
# print(nnp__i__p____q____q)
# nnp__i__p____q____q.evaluate(c2)

In [7]:
a, b, c, d, e = Prop("a"), Prop("b"), Prop("c"), Prop("d"), Prop("e")
alpha = Implies(Implies(Implies(Implies(e, d), c), b), a)
beta = Implies(Implies(e, Implies(Implies(d, a), b)), Implies(Implies(c, b), a))

c1 = KripkiState(["a", "b", "c", "d", "e"], [])
c0 = KripkiState([], [c1])

print("c1")
print(alpha, alpha.evaluate(c1))
print(beta, beta.evaluate(c1))

print("c0")
print(alpha, alpha.evaluate(c0))
print(beta, beta.evaluate(c0))

c1
((((e → d) → c) → b) → a) True
((e → ((d → a) → b)) → ((c → b) → a)) True
c0
((((e → d) → c) → b) → a) False
((e → ((d → a) → b)) → ((c → b) → a)) False


In [8]:
c1 = KripkiState(["a", "b", "c", "d", "e"], [])
c0 = KripkiState([], [c1])

print("c1")
print(alpha, alpha.evaluate(c1))
print(beta, beta.evaluate(c1))

print("c0")
print(alpha, alpha.evaluate(c0))
print(beta, beta.evaluate(c0))

c1
((((e → d) → c) → b) → a) True
((e → ((d → a) → b)) → ((c → b) → a)) True
c0
((((e → d) → c) → b) → a) False
((e → ((d → a) → b)) → ((c → b) → a)) False


In [9]:
from itertools import chain, combinations


def powerset(iterable):
    xs = list(iterable)
    return chain.from_iterable(combinations(xs, n) for n in range(len(xs) + 1))

In [10]:
def is_valid(c0_vars, c1_vars):
    c1 = KripkiState(c1_vars, [])
    c0 = KripkiState(c0_vars, [c1])
    return alpha.evaluate(c0) == beta.evaluate(c0)

In [11]:
all_vars = ["a", "b", "c", "d", "e"]
for c1_vars in powerset(all_vars):
    for c0_vars in powerset(c1_vars):
        if not is_valid(c0_vars, c1_vars):
            c1 = KripkiState(c1_vars, [])
            c0 = KripkiState(c0_vars, [c1])
            print('c0_vars:', c0_vars)
            print('c1_vars:', c1_vars)
            print(alpha, alpha.evaluate(c0))
            print(beta, beta.evaluate(c0))
            print('=' * 120)

c0_vars: ()
c1_vars: ('a', 'b', 'e')
((((e → d) → c) → b) → a) True
((e → ((d → a) → b)) → ((c → b) → a)) False
c0_vars: ()
c1_vars: ('a', 'd', 'e')
((((e → d) → c) → b) → a) False
((e → ((d → a) → b)) → ((c → b) → a)) True
c0_vars: ('d',)
c1_vars: ('a', 'd', 'e')
((((e → d) → c) → b) → a) False
((e → ((d → a) → b)) → ((c → b) → a)) True
c0_vars: ('e',)
c1_vars: ('a', 'd', 'e')
((((e → d) → c) → b) → a) False
((e → ((d → a) → b)) → ((c → b) → a)) True
c0_vars: ('d', 'e')
c1_vars: ('a', 'd', 'e')
((((e → d) → c) → b) → a) False
((e → ((d → a) → b)) → ((c → b) → a)) True
c0_vars: ()
c1_vars: ('a', 'b', 'c', 'e')
((((e → d) → c) → b) → a) True
((e → ((d → a) → b)) → ((c → b) → a)) False
c0_vars: ('e',)
c1_vars: ('a', 'b', 'd', 'e')
((((e → d) → c) → b) → a) False
((e → ((d → a) → b)) → ((c → b) → a)) True


In [17]:
p, q = Prop("p"), Prop("q")

c1 = KripkiState(["q"], [])
c2 = KripkiState([], [c1])
c0 = KripkiState([], [c1, c2])

alpha = Implies(Neg(p), q)
beta = Neg(Neg(Or(p, q)))
beta__i__alpha = Implies(beta, alpha)
beta__i__alpha

(¬¬(p ∨ q) → (¬p → q))

In [18]:
beta__i__alpha.evaluate(c0)

False

In [36]:
p, q, r = Prop("p"), Prop("q"), Prop("r")

phi = Or(Or(p, Neg(q)), Implies(p, q))
# phi = Implies(Implies(Implies(p, q), r), Implies(Implies(Implies(r, p), r), r))

for c1_vars in powerset(["p", "q", "r"]):
    for c0_vars in powerset(c1_vars):
        c1 = KripkiState(c1_vars, [])
        c0 = KripkiState(c0_vars, [c1])
        phi_val = phi.evaluate(c0)
        print(f'{phi} = {phi_val}')
        print(c0_vars, c1_vars)
        print('=' * 120)

((p ∨ ¬q) ∨ (p → q)) = True
() ()
((p ∨ ¬q) ∨ (p → q)) = True
() ('p',)
((p ∨ ¬q) ∨ (p → q)) = True
('p',) ('p',)
((p ∨ ¬q) ∨ (p → q)) = True
() ('q',)
((p ∨ ¬q) ∨ (p → q)) = True
('q',) ('q',)
((p ∨ ¬q) ∨ (p → q)) = True
() ('r',)
((p ∨ ¬q) ∨ (p → q)) = True
('r',) ('r',)
((p ∨ ¬q) ∨ (p → q)) = True
() ('p', 'q')
((p ∨ ¬q) ∨ (p → q)) = True
('p',) ('p', 'q')
((p ∨ ¬q) ∨ (p → q)) = True
('q',) ('p', 'q')
((p ∨ ¬q) ∨ (p → q)) = True
('p', 'q') ('p', 'q')
((p ∨ ¬q) ∨ (p → q)) = True
() ('p', 'r')
((p ∨ ¬q) ∨ (p → q)) = True
('p',) ('p', 'r')
((p ∨ ¬q) ∨ (p → q)) = True
('r',) ('p', 'r')
((p ∨ ¬q) ∨ (p → q)) = True
('p', 'r') ('p', 'r')
((p ∨ ¬q) ∨ (p → q)) = True
() ('q', 'r')
((p ∨ ¬q) ∨ (p → q)) = True
('q',) ('q', 'r')
((p ∨ ¬q) ∨ (p → q)) = True
('r',) ('q', 'r')
((p ∨ ¬q) ∨ (p → q)) = True
('q', 'r') ('q', 'r')
((p ∨ ¬q) ∨ (p → q)) = True
() ('p', 'q', 'r')
((p ∨ ¬q) ∨ (p → q)) = True
('p',) ('p', 'q', 'r')
((p ∨ ¬q) ∨ (p → q)) = True
('q',) ('p', 'q', 'r')
((p ∨ ¬q) ∨ (p → q)) = Tr

In [37]:
c1_vars = ['p', 'r', 'q']
c0_vars = []

c1 = KripkiState(c1_vars, [])
c0 = KripkiState(c0_vars, [c1])
phi.evaluate(c0)

True

In [38]:
#  c0
# /  \
# c1 c2

for c1_vars in powerset(["p", "q", "r"]):
    for c2_vars in powerset(["p", "q", "r"]):
        children_vars = list(set(c1_vars) & set(c2_vars))
        for c0_vars in powerset(children_vars):
            c1 = KripkiState(c1_vars, [])
            c2 = KripkiState(c2_vars, [])
            c0 = KripkiState(c0_vars, [c1, c2])
            phi_val = phi.evaluate(c0)
            if not phi_val:
                print(f'{phi} = {phi_val}')
                print('c0_vars:', c0_vars, 'c1_vars:', c1_vars, 'c2_vars:', c2_vars)
                print('=' * 120)

((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: () c1_vars: ('p',) c2_vars: ('q',)
((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: () c1_vars: ('p',) c2_vars: ('p', 'q')
((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: () c1_vars: ('p',) c2_vars: ('q', 'r')
((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: () c1_vars: ('p',) c2_vars: ('p', 'q', 'r')
((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: () c1_vars: ('q',) c2_vars: ('p',)
((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: () c1_vars: ('q',) c2_vars: ('p', 'r')
((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: () c1_vars: ('p', 'q') c2_vars: ('p',)
((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: () c1_vars: ('p', 'q') c2_vars: ('p', 'r')
((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: () c1_vars: ('p', 'r') c2_vars: ('q',)
((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: () c1_vars: ('p', 'r') c2_vars: ('p', 'q')
((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: () c1_vars: ('p', 'r') c2_vars: ('q', 'r')
((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: ('r',) c1_vars: ('p', 'r') c2_vars: ('q', 'r')
((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: () c1_vars: ('p', 'r'

In [42]:
c0_vars = []
c1_vars = ['p']
c2_vars = ['q']
c3_vars = []

c1 = KripkiState(c1_vars, [])
c2 = KripkiState(c2_vars, [])
c3 = KripkiState(c3_vars, [])
c0 = KripkiState(c0_vars, [c1, c2, c3])
phi_val = phi.evaluate(c0)
print(f'{phi} = {phi_val}')
print('c0_vars:', c0_vars, 'c1_vars:', c1_vars, 'c2_vars:', c2_vars)
print('=' * 120)

((p ∨ ¬q) ∨ (p → q)) = False
c0_vars: [] c1_vars: ['p'] c2_vars: ['q']


In [43]:
phi = Or(Neg(p), Neg(Neg(p)))

c1 = KripkiState(["p"], [])
c2 = KripkiState([], [])
c0 = KripkiState([], [c1, c2])

phi.evaluate(c0)

False

In [44]:
phi = Or(Implies(p, q), Implies(q, p))

c1 = KripkiState(["p"], [])
c2 = KripkiState(["q"], [])
c0 = KripkiState([], [c1, c2])

phi.evaluate(c0)

False

In [51]:
phi = Iff(Implies(p, q), Or(Neg(p), q))

c1 = KripkiState(["p", "q"], [])
c0 = KripkiState([], [c1])

phi.evaluate(c0)

False

In [60]:
# (¬p → q ∨ r) → ((¬p → q) ∨ (¬p → r))

phi = Implies(Implies(Neg(p), Or(q, r)), Or(Implies(Neg(p), q), Implies(Neg(p), r)))
print(phi)

c1 = KripkiState(["r"], [])
c2 = KripkiState(["q"], [])
c3 = KripkiState(["p"], [])
c0 = KripkiState([], [c1, c2, c3])

phi.evaluate(c0)

((¬p → (q ∨ r)) → ((¬p → q) ∨ (¬p → r)))


False